# Task B — FILM (large-motion VFI) + RIFE ensemble

The public test is shifted toward **large motion** (delta_s=2, high flow) — exactly where
RIFE-HDv3 ghosts and our score plateaued at ~48.5. **FILM** is designed for large-motion
interpolation (frames seconds apart). This notebook loads FILM + RIFE, compares them and
their ensemble on an **honest scene-disjoint** val split (real contest score), and submits
the winner. Safe-mean fallback on the highest-motion tail.

**Attach:** competition dataset · RIFE (repo + train_log) · FILM saved_model dataset.
**Setup:** GPU T4 ×2, Internet ON.

## 1. Imports, paths, GPU memory growth (so TF + Torch coexist)

In [ ]:
import os, glob, json, math, random, sys, shutil
from pathlib import Path
import numpy as np, cv2
from PIL import Image
from tqdm.auto import tqdm

import tensorflow as tf
for g in tf.config.list_physical_devices('GPU'):       # don't let TF grab all VRAM
    try: tf.config.experimental.set_memory_growth(g, True)
    except Exception: pass
import torch, torch.nn as nn, torch.nn.functional as F
random.seed(0); np.random.seed(0); torch.manual_seed(0)
device='cuda' if torch.cuda.is_available() else 'cpu'
print('torch', torch.__version__, '| tf', tf.__version__, '| device', device)

def find_split_root(want):
    for m in sorted(glob.glob(f'/kaggle/input/**/{want}', recursive=True)):
        if os.path.isdir(m): return os.path.dirname(m)
    return None
TRAIN_ROOT=(find_split_root('train') or '')+'/train'
_eb=find_split_root('test'); TEST_ROOT=(_eb+'/test') if _eb else None
print('TRAIN_ROOT:',TRAIN_ROOT,os.path.isdir(TRAIN_ROOT)); print('TEST_ROOT :',TEST_ROOT)

## 2. Load FILM
Tries, in order: (a) a local saved_model dataset, (b) download from TF-Hub (needs
Internet ON — no dataset upload required).

In [ ]:
film=None
# (a) local saved_model dataset, if attached
_pb=glob.glob('/kaggle/input/**/saved_model.pb',recursive=True)
if _pb:
    FILM_DIR=str(Path(_pb[0]).parent); print('local FILM_DIR:',FILM_DIR)
    film=tf.saved_model.load(FILM_DIR)
# (b) download from TF-Hub
if film is None:
    try:
        import tensorflow_hub as hub
    except Exception:
        import subprocess,sys as _s; subprocess.run([_s.executable,'-m','pip','install','-q','tensorflow_hub']); import tensorflow_hub as hub
    print('downloading FILM from TF-Hub (needs Internet ON)...')
    film=hub.load('https://tfhub.dev/google/film/1')
print('FILM loaded. (inputs: x0/x1/time)')

def _pad64_np(a):                       # HxWx3 float -> padded, (h,w)
    h,w=a.shape[:2]; ph=((h-1)//64+1)*64; pw=((w-1)//64+1)*64
    return np.pad(a,((0,ph-h),(0,pw-w),(0,0)),mode='reflect'), h, w

def film_mid(i0u, i1u, t=0.5):          # uint8 HxWx3 -> float HxWx3 in 0..255
    a0,h,w=_pad64_np(i0u.astype(np.float32)/255.); a1,_,_=_pad64_np(i1u.astype(np.float32)/255.)
    inp={'x0':tf.constant(a0[None]),'x1':tf.constant(a1[None]),'time':tf.constant([[t]],tf.float32)}
    out=film(inp)['image'][0].numpy()[:h,:w]
    return np.clip(out,0,1)*255.

## 3. Load RIFE (for comparison / ensemble)

In [ ]:
try:
    w_dir=str(Path(glob.glob('/kaggle/input/**/flownet.pkl',recursive=True)[0]).parent)
    _wl=glob.glob('/kaggle/input/**/model/warplayer.py',recursive=True)
    if _wl: sys.path.insert(0,str(Path(_wl[0]).parent.parent))
    sys.path.insert(0,str(Path(w_dir).parent)); sys.path.insert(0,w_dir)
    from train_log.RIFE_HDv3 import Model
    _m=Model(); _m.load_model(w_dir,-1); _m.flownet.to(device).eval(); flownet=_m.flownet
    ftw=glob.glob('/kaggle/input/**/flownet_ft.pkl',recursive=True)+glob.glob('/kaggle/working/flownet_ft.pkl')
    if ftw: flownet.load_state_dict(torch.load(ftw[0],map_location=device)); print('loaded ft weights',ftw[0])
    print('RIFE ready')
except Exception as e:
    flownet=None; print('RIFE not available (FILM-only mode):', e)

@torch.no_grad()
def rife_scaled(i0u,i1u,scale=0.5,tta=True):
    if flownet is None: return None
    m=64*math.ceil(1.0/scale)
    def padm(t):
        _,_,h,w=t.shape; ph=((h-1)//m+1)*m; pw=((w-1)//m+1)*m
        return F.pad(t,(0,pw-w,0,ph-h),mode='replicate'),h,w
    def run(a0,a1):
        i0=torch.from_numpy(np.ascontiguousarray(a0)).permute(2,0,1)[None].float().to(device)/255.
        i1=torch.from_numpy(np.ascontiguousarray(a1)).permute(2,0,1)[None].float().to(device)/255.
        x0,h,w=padm(i0); x1,_,_=padm(i1)
        sl=[16/scale,8/scale,4/scale,2/scale,1/scale]
        _,_,merged=flownet(torch.cat((x0,x1),1),0.5,sl)
        return merged[-1][:,:, :h,:w].clamp(0,1)[0].permute(1,2,0).cpu().numpy()*255
    p=run(i0u,i1u)
    if tta: pf=run(i0u[:,::-1],i1u[:,::-1])[:,::-1]; p=0.5*(p+pf)
    return p

## 4. Helpers + sanity PSNR on a few train samples

In [ ]:
def psnr01(a,b):
    a=np.clip(a,0,1).astype(np.float64); b=np.clip(b,0,1).astype(np.float64)
    m=np.mean((a-b)**2); return 99.0 if m<1e-9 else 20*np.log10(1/np.sqrt(m))
def score_of(p): return (min(max(p,10),30)-10)/20*100
def load_uu(sd):
    meta=json.loads((sd/'meta.json').read_text()); cam=meta['target_camera']
    i0=np.array(Image.open(sd/'input'/'t0'/f'{cam}.jpg').convert('RGB')); i1=np.array(Image.open(sd/'input'/'t1'/f'{cam}.jpg').convert('RGB'))
    return meta,cam,i0,i1
def motion_score(i0u,i1u):
    fl=cv2.DISOpticalFlow_create(cv2.DISOPTICAL_FLOW_PRESET_MEDIUM).calc(
        cv2.cvtColor(i0u,cv2.COLOR_RGB2GRAY),cv2.cvtColor(i1u,cv2.COLOR_RGB2GRAY),None)
    return float(np.median(np.sqrt((fl**2).sum(-1))))

# quick smoke test on 3 train samples
for sd in sorted(p for p in Path(TRAIN_ROOT).iterdir() if p.is_dir())[:3]:
    meta,cam,i0u,i1u=load_uu(sd)
    gt=np.array(Image.open(sd/'target'/f'{cam}.jpg').convert('RGB')).astype(np.float32)/255.
    pf=film_mid(i0u,i1u)/255.; line=f'{sd.name[:18]} delta={meta.get("delta_s")} FILM={psnr01(pf,gt):.2f}'
    if flownet is not None:
        pr=rife_scaled(i0u,i1u)/255.; line+=f' RIFE={psnr01(pr,gt):.2f}'
    print(line)

## 5. Honest scene-disjoint comparison (per delta_s)
FILM vs RIFE vs ensembles, scored with the real contest metric, split by `delta_s`
(the large-motion half is the interesting one).

In [ ]:
# build scene-disjoint val
scenes={}
for p in sorted(Path(TRAIN_ROOT).iterdir()):
    if not p.is_dir(): continue
    s=json.loads((p/'meta.json').read_text()).get('scene','?'); scenes.setdefault(s,[]).append(p)
scl=sorted(scenes); random.seed(11); random.shuffle(scl); val_sc=set(scl[:max(1,len(scl)//5)])
val=[p for s in val_sc for p in scenes[s]]; random.shuffle(val); val=val[:140]
print(f'{len(scl)} scenes | val on {len(val)} samples')

from collections import defaultdict
cand=['film','rife','0.5film+0.5rife','0.7film+0.3rife','0.3film+0.7rife']
acc=defaultdict(lambda: defaultdict(float)); cnt=defaultdict(int)
for sd in tqdm(val):
    meta,cam,i0u,i1u=load_uu(sd); d=meta.get('delta_s',1.0)
    gt=np.array(Image.open(sd/'target'/f'{cam}.jpg').convert('RGB')).astype(np.float32)/255.
    fm=film_mid(i0u,i1u)/255.
    preds={'film':fm}
    if flownet is not None:
        rf=rife_scaled(i0u,i1u)/255.
        preds.update({'rife':rf,'0.5film+0.5rife':0.5*fm+0.5*rf,'0.7film+0.3rife':0.7*fm+0.3*rf,'0.3film+0.7rife':0.3*fm+0.7*rf})
    cnt[d]+=1
    for k,pr in preds.items(): acc[d][k]+=score_of(psnr01(pr,gt))
for d in sorted(acc):
    print(f'delta_s={d} (n={cnt[d]}):')
    for k,v in sorted(acc[d].items(),key=lambda kv:-kv[1]): print(f'  {k:16s} {v/cnt[d]:.3f}')
# overall best method
tot=defaultdict(float); N=sum(cnt.values())
for d in acc:
    for k,v in acc[d].items(): tot[k]+=v
print('\nOVERALL:');
for k,v in sorted(tot.items(),key=lambda kv:-kv[1]): print(f'  {k:16s} {v/N:.3f}')
BEST=max(tot,key=tot.get); print('BEST:',BEST)

## 6. Submission with the winning method (q100)

In [ ]:
def predict(i0u,i1u,method):
    fm=film_mid(i0u,i1u)
    if method=='film' or flownet is None: return fm
    rf=rife_scaled(i0u,i1u)
    return {'rife':rf,'0.5film+0.5rife':0.5*fm+0.5*rf,'0.7film+0.3rife':0.7*fm+0.3*rf,'0.3film+0.7rife':0.3*fm+0.7*rf}[method]

out='/kaggle/working/submission'
if os.path.exists(out): shutil.rmtree(out)
os.makedirs(out,exist_ok=True)
for sd in tqdm(sorted(p for p in Path(TEST_ROOT).iterdir() if p.is_dir())):
    meta,cam,i0u,i1u=load_uu(sd)
    pred=np.clip(predict(i0u,i1u,BEST),0,255).round().astype(np.uint8)
    od=Path(out)/sd.name; od.mkdir(parents=True,exist_ok=True)
    Image.fromarray(pred).save(od/'pred.jpg',quality=100,subsampling=0)
shutil.make_archive('/kaggle/working/submission','zip',out)
print('submission ready | method:',BEST)